## 1. Импорт зависимостей

In [2]:
import sys
from pathlib import Path
import polars as pl
import pandas as pd
import json
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from match import CONFIG_DIR, resolve_project_path, normalize_attributes 

with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="prepare_data")

pl.Config.set_tbl_rows(-1)       # показывать все строки
pl.Config.set_tbl_cols(-1)       # показывать все столбцы
pl.Config.set_fmt_str_lengths(1000)  # не обрезать длинные строки
pl.Config.set_tbl_width_chars(200)   # ширина таблицы

polars.config.Config

## 2. Чтение конфига

In [3]:
cfg

{'path': {'items_human_path': 'data/items_human_normalized.parquet', 'matches_human_path': 'data/matches.parquet', 'items_human_normalized_path': 'data/items_human_normalized.parquet', 'synonyms': 'data/synonyms_df.parquet', 'unique_attributes': 'data/untouched_stats.parquet'}}

## 3. Чтение датасета

In [4]:
df_human = pl.read_parquet(resolve_project_path(cfg.path.items_human_path))
df_matches = pl.read_parquet(resolve_project_path(cfg.path.matches_human_path))
synonyms_df = pl.read_parquet(resolve_project_path(cfg.path.synonyms))
unique_attributes = pl.read_parquet(resolve_project_path(cfg.path.unique_attributes))
df_human.head(1), df_matches.head(1), synonyms_df.head(1), unique_attributes.head(1)

(shape: (1, 5)
 ┌─────┬───────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────┬────────────┬───────────────────────────────────────────────────────────┐
 │ id  ┆ name                                                      ┆ attributes                                                ┆ category   ┆ normalized_attributes                                     │
 │ --- ┆ ---                                                       ┆ ---                                                       ┆ ---        ┆ ---                                                       │
 │ i64 ┆ str                                                       ┆ str                                                       ┆ str        ┆ str                                                       │
 ╞═════╪═══════════════════════════════════════════════════════════╪═══════════════════════════════════════════════════════════╪════════════╪════════════════════════════════════

## 3.2 Разбор карточек которые совпадают

In [5]:
df_matches.filter(pl.col("target") == 1).head()

id1,id2,target
i64,i64,f64
476,841813632218,1.0
525,13777,1.0
622,360777364649,1.0
706,352187426187,1.0
753,747324374388,1.0


In [6]:
row = df_matches.filter(pl.col("target") == 1).sample(1)
id1, id2 = row.select("id1", "id2").row(0)


item1 = (
    df_human
    .filter(pl.col("id") == id1)
    .select("id", "name", "normalized_attributes")
    .row(0, named=True)
)

item2 = (
    df_human
    .filter(pl.col("id") == id2)
    .select("id", "name", "normalized_attributes")
    .row(0, named=True)
)


attrs1 = json.loads(item1["normalized_attributes"])
attrs2 = json.loads(item2["normalized_attributes"])

all_keys = sorted(set(attrs1) | set(attrs2))


names = pl.DataFrame({
    "id": [item1["id"], item2["id"]],
    "name": [item1["name"], item2["name"]],
})

comparison = (
    pl.DataFrame({
        "attribute": all_keys,
        f"value_{id1}": [attrs1.get(key) for key in all_keys],
        f"value_{id2}": [attrs2.get(key) for key in all_keys],
    })
    .with_columns(
        (
            pl.col(f"value_{id1}").fill_null("<нет>")
            == pl.col(f"value_{id2}").fill_null("<нет>")
        ).alias("equal")
    )
    .sort(["equal", "attribute"], descending=[True, False])
)

display(names)
display(comparison)

id,name
i64,str
17179930565,"""соус kikkoman соевый 150 г"""
60129636816,"""соус kikkoman соевый, 150мл"""


attribute,value_17179930565,value_60129636816,equal
str,str,str,bool
"""тип соуса""","""соевый""","""соевый""",true
"""белки""","""10.0 г""",null,false
"""бренд""",null,"""kikkoman""",false
"""вес, кг""","""0.15""",null,false
"""калорийность""","""75.0 ккал""",null,false
"""кол в одном товаре""",null,"""1""",false
"""количество""","""150 г""",null,false
"""количество в упаковке""","""1 шт""",null,false
"""макс. температура""",null,"""25""",false


## 4. Дисбаланс класса 1/4

In [7]:
attrs = json.loads(df_human.filter(pl.col("id") == id1).select("attributes").to_series().to_list()[0])
attrs.keys()

dict_keys(['вид соуса', 'назначение', 'вес', 'основной вкус', 'кухни мира', 'вид упаковки', 'количество в упаковке', 'состав', 'белки', 'углеводы', 'калорийность', 'условия хранения', 'срок хранения', 'условия хранения 2', 'срок хранения 2', 'производитель', 'страна', 'штук в упаковке', 'размер'])

In [8]:
(
    df_matches
    .group_by("target")
    .agg(pl.count().alias("count"))
    .with_columns(
        (pl.col("count") / pl.sum("count")).alias("ratio")
    )
    .sort("target")
)

C:\Users\Samoylov_Nikita\AppData\Local\Temp\ipykernel_28772\3475110605.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("count"))


target,count,ratio
f64,u32,f64
0.0,271764,0.743227
1.0,93890,0.256773


## 5. Разбор датасета товаров

In [9]:
df_human.head()

id,name,attributes,category,normalized_attributes
i64,str,str,str,str
197,"""victor reinz прокладка впускного коллектора арт. 703315700""","""{""артикул"":""703315700"",""бренд"":""victor reinz"",""партномер (артикул производителя)"":""703315700"",""тип"":""прокладка двигателя"",""вид техники"":""легковые автомобили""}""","""Автотовары""","""{""артикул"": ""703315700"", ""бренд"": ""victor reinz"", ""партномер (артикул производителя)"": ""703315700"", ""тип"": ""прокладка двигателя"", ""тип системы"": ""легковые автомобили""}"""
415,"""stellox диск тормозной, арт. 8500892sx""","""{""артикул"":""stellox_8500892sx"",""место установки"":""передние"",""толщина тормозного диска, мм"":""28"",""бренд"":""stellox"",""oem-номер"":""3050086; a9064210012; 2e0615301; 68006716aa; 9064210012; 9064210212; 9064210012s; 9064210112"",""партномер (артикул производителя)"":""8500892sx"",""тип"":""диск тормозной"",""тип тормозного диска"":""вентилируемый"",""вид техники"":""легковые автомобили""}""","""Автотовары""","""{""артикул"": ""stellox_8500892sx"", ""система установки"": ""передние"", ""толщина тормозного диска, мм"": ""28"", ""бренд"": ""stellox"", ""oem-номер"": ""3050086; a9064210012; 2e0615301; 68006716aa; 9064210012; 9064210212; 9064210012s; 9064210112"", ""партномер (артикул производителя)"": ""8500892sx"", ""тип"": ""диск тормозной"", ""тип тормозного диска"": ""вентилируемый"", ""тип системы"": ""легковые автомобили""}"""
427,"""комплект подшипника ступицы колеса lynxauto""","""{""артикул производителя"":"""",""бренд"":""lynxauto"",""примечание"":""внешний вид изделия может отличаться от фотографий"",""цена за"":""1 шт."",""код товара"":""600000256748""}""","""Автотовары""","""{""артикул производителя"": """", ""бренд"": ""lynxauto"", ""примечание"": ""внешний вид изделия может отличаться от фотографий"", ""плата за"": ""1 шт."", ""код товара"": ""600000256748""}"""
1027,"""kraft подшипник ступицы, арт. kt204632, 1 шт.""","""{""артикул"":""112097-01"",""бренд"":""kraft"",""количество в упаковке, шт"":""1"",""партномер (артикул производителя)"":""kt204632"",""тип"":""подшипник ступицы""}""","""Автотовары""","""{""артикул"": ""112097-01"", ""бренд"": ""kraft"", ""количество в упаковке, шт"": ""1"", ""партномер (артикул производителя)"": ""kt204632"", ""тип"": ""подшипник ступицы""}"""
2639,"""фильтр салонный skoda fabia 00-""","""{""альтернативные артикулы товара"":""8104400xkz96a;50013702;ac0110c;50013941;9.7.84;la120;if-3019;afc1076;ac9403;ca-49010;6q0820367b;fs089;lak120;sab 123;1987432057;1010-026;6q0819653;6q0819653b;9.7.147;ac0110c;fs-089;dfc2545;if3019k;if3019p;sa1123;afw1076;gb9892c;fcr21f061;jdacx055;lac-1006;7110221sx;7110247sx;1cf031;nfe-2389;k1079;cuk2545;wc4015;7110543sx;st-6q0820367b;nf6123c;sa 1123;pf2123;mc-e4072;nf6123;1cf041;pf2041;lac1006c;8104400xkz96a\n"",""артикул"":""cu2545"",""бренд"":""sat"",""страна-изготовитель"":""китай"",""oem-номер"":""8104400xkz96a;50013702;ac0110c;50013941;9.7.84;la120;if-3019;afc1076;ac9403;ca-49010;6q0820367b;fs089;lak120;sab 123;1987432057;1010-026;6q0819653;6q0819653b;9.7.147;ac0110c;fs-089;dfc2545;if3019k;if3019p;sa1123;afw1076;gb9892c;fcr21f061;jdacx055;lac-1006;7110221sx;7110247sx;1cf031;nfe-2389;k1079;cuk2545;wc4015;7110543sx;st-6q0820367b;nf6123c;sa 1123;pf2123;mc-e4072;nf6123;1cf041;pf2041;lac1006c;8104400xkz96a\n"",""комплектация"":""фильтр салона для авто 1 шт"",""партномер (…","""Автотовары""","""{""альтернативные артикулы товара"": ""8104400xkz96a;50013702;ac0110c;50013941;9.7.84;la120;if-3019;afc1076;ac9403;ca-49010;6q0820367b;fs089;lak120;sab 123;1987432057;1010-026;6q0819653;6q0819653b;9.7.147;ac0110c;fs-089;dfc2545;if3019k;if3019p;sa1123;afw1076;gb9892c;fcr21f061;jdacx055;lac-1006;7110221sx;7110247sx;1cf031;nfe-2389;k1079;cuk2545;wc4015;7110543sx;st-6q0820367b;nf6123c;sa 1123;pf2123;mc-e4072;nf6123;1cf041;pf2041;lac1006c;8104400xkz96a"", ""артикул"": ""cu2545"", ""бренд"": ""sat"", ""страна-производитель"": ""китай"", ""oem-номер"": ""8104400xkz96a;50013702;ac0110c;500

## 6. Нормализация атрибутов

In [ ]:
df_norm_human = normalize_attributes(df_human, resolve_project_path(cfg.path.synonyms), resolve_project_path(cfg.path.unique_attributes))

2026-08-16 20:38:51.772 | INFO     | match.normalization:normalize_attributes:446 - Starting attribute normalization: rows=711304, source_column='attributes', output_column='normalized_attributes'
2026-08-16 20:38:53.800 | INFO     | match.normalization:normalize_attributes:470 - Loaded normalization metadata: synonym_replacements=946, unique_attributes=26552
2026-08-16 20:38:53.801 | INFO     | match.normalization:normalize_attributes:489 - Using process-based normalization: n_jobs=2, chunk_size=5000, chunks=143


In [ ]:
df_norm_human.head(1)